# Realistic Human Face Generation using GAN (DCGAN)
Generate synthetic 64x64 human faces from random noise using a Generator and Discriminator trained adversarially on CelebA.

**Run on:** Google Colab (Runtime > Change runtime type > T4 GPU)

## 1. Setup

In [ ]:
!pip install -q kagglehub

import os, glob, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.utils as vutils
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---- Config ----
IMG_SIZE    = 64
LATENT_DIM  = 100
BATCH_SIZE  = 128
EPOCHS      = 30
LR          = 2e-4
BETA1       = 0.5
MAX_IMAGES  = 100_000   # use None for the full 202k images
OUT_DIR     = "gan_outputs"
os.makedirs(f"{OUT_DIR}/samples", exist_ok=True)
os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True)
os.makedirs("results", exist_ok=True)

## 2. Load and preprocess the dataset (crop, resize, normalize)
CelebA from Kaggle. Each image is center-cropped to the face region (148x148), resized to 64x64, and normalized to [-1, 1] to match the Generator's `tanh` output.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("jessicali9530/celeba-dataset")
files = sorted(glob.glob(os.path.join(path, "**", "*.jpg"), recursive=True))
if MAX_IMAGES: files = files[:MAX_IMAGES]
print("Images:", len(files))

transform = transforms.Compose([
    transforms.CenterCrop(148),
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

class FaceDataset(Dataset):
    def __init__(self, files, transform): self.files, self.transform = files, transform
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        return self.transform(Image.open(self.files[i]).convert("RGB"))

loader = DataLoader(FaceDataset(files, transform), batch_size=BATCH_SIZE,
                    shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

real_batch = next(iter(loader))
plt.figure(figsize=(7, 7)); plt.axis("off"); plt.title("Real training faces")
plt.imshow(vutils.make_grid(real_batch[:64], padding=2, normalize=True).permute(1, 2, 0))
plt.show()

## 3. Build the Generator and Discriminator

In [ ]:
class Generator(nn.Module):
    """Latent vector (100) -> 3x64x64 image"""
    def __init__(self, z=LATENT_DIM, f=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z, f*8, 4, 1, 0, bias=False), nn.BatchNorm2d(f*8), nn.ReLU(True),      # 4x4
            nn.ConvTranspose2d(f*8, f*4, 4, 2, 1, bias=False), nn.BatchNorm2d(f*4), nn.ReLU(True),    # 8x8
            nn.ConvTranspose2d(f*4, f*2, 4, 2, 1, bias=False), nn.BatchNorm2d(f*2), nn.ReLU(True),    # 16x16
            nn.ConvTranspose2d(f*2, f, 4, 2, 1, bias=False),   nn.BatchNorm2d(f),   nn.ReLU(True),    # 32x32
            nn.ConvTranspose2d(f, 3, 4, 2, 1, bias=False), nn.Tanh(),                                 # 64x64
        )
    def forward(self, z): return self.net(z.view(z.size(0), -1, 1, 1))

class Discriminator(nn.Module):
    """Image 3x64x64 -> real/fake logit"""
    def __init__(self, f=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, f, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),                                        # 32x32
            nn.Conv2d(f, f*2, 4, 2, 1, bias=False), nn.BatchNorm2d(f*2), nn.LeakyReLU(0.2, True),                 # 16x16
            nn.Conv2d(f*2, f*4, 4, 2, 1, bias=False), nn.BatchNorm2d(f*4), nn.LeakyReLU(0.2, True),               # 8x8
            nn.Conv2d(f*4, f*8, 4, 2, 1, bias=False), nn.BatchNorm2d(f*8), nn.LeakyReLU(0.2, True),               # 4x4
            nn.Conv2d(f*8, 1, 4, 1, 0, bias=False),                                                               # 1x1
        )
    def forward(self, x): return self.net(x).view(-1)

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)): nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)

class GAN(nn.Module):
    """Combines Generator and Discriminator into one model."""
    def __init__(self):
        super().__init__()
        self.G, self.D = Generator(), Discriminator()
        self.G.apply(init_weights); self.D.apply(init_weights)

gan = GAN().to(device)
G, D = gan.G, gan.D
print(f"Generator params:     {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")

## 4. Train with adversarial learning
Each step: (1) train **D** on real vs. generated images, (2) train **G** to fool D. A fixed noise batch is used so progress is comparable across epochs.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optD = torch.optim.Adam(D.parameters(), lr=LR, betas=(BETA1, 0.999))
optG = torch.optim.Adam(G.parameters(), lr=LR, betas=(BETA1, 0.999))

fixed_noise = torch.randn(64, LATENT_DIM, device=device)
REAL_LABEL, FAKE_LABEL = 0.9, 0.0   # one-sided label smoothing

G_losses, D_losses, D_real_scores, D_fake_scores = [], [], [], []

def save_samples(epoch):
    G.eval()
    with torch.no_grad(): fake = G(fixed_noise).cpu()
    G.train()
    vutils.save_image(fake, f"{OUT_DIR}/samples/epoch_{epoch:03d}.png", nrow=8, normalize=True, padding=2)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    for real in loader:
        real = real.to(device, non_blocking=True)
        b = real.size(0)

        # --- Train Discriminator ---
        D.zero_grad()
        out_real = D(real)
        loss_real = criterion(out_real, torch.full((b,), REAL_LABEL, device=device))
        noise = torch.randn(b, LATENT_DIM, device=device)
        fake = G(noise)
        out_fake = D(fake.detach())
        loss_fake = criterion(out_fake, torch.full((b,), FAKE_LABEL, device=device))
        (loss_real + loss_fake).backward()
        optD.step()

        # --- Train Generator ---
        G.zero_grad()
        out = D(fake)
        lossG = criterion(out, torch.ones(b, device=device))
        lossG.backward()
        optG.step()

        D_losses.append((loss_real + loss_fake).item()); G_losses.append(lossG.item())
        D_real_scores.append(torch.sigmoid(out_real).mean().item())
        D_fake_scores.append(torch.sigmoid(out_fake).mean().item())

    save_samples(epoch)
    torch.save({"G": G.state_dict(), "D": D.state_dict()}, f"{OUT_DIR}/checkpoints/gan_last.pt")
    print(f"Epoch {epoch:02d}/{EPOCHS} | D loss {np.mean(D_losses[-len(loader):]):.3f} | "
          f"G loss {np.mean(G_losses[-len(loader):]):.3f} | D(x) {np.mean(D_real_scores[-len(loader):]):.2f} | "
          f"D(G(z)) {np.mean(D_fake_scores[-len(loader):]):.2f} | {time.time()-t0:.0f}s")

## 5. Monitor losses

In [ ]:
def smooth(x, k=100): return np.convolve(x, np.ones(k)/k, mode="valid")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(smooth(G_losses), label="Generator"); ax[0].plot(smooth(D_losses), label="Discriminator")
ax[0].set_title("Training losses (smoothed)"); ax[0].set_xlabel("Iteration"); ax[0].set_ylabel("Loss"); ax[0].legend()
ax[1].plot(smooth(D_real_scores), label="D(x) on real"); ax[1].plot(smooth(D_fake_scores), label="D(G(z)) on fake")
ax[1].set_title("Discriminator confidence"); ax[1].set_xlabel("Iteration"); ax[1].legend()
plt.tight_layout(); plt.savefig("results/loss_curves.png", dpi=150); plt.show()

## 6. Compare results over time

In [ ]:
epochs_to_show = sorted(set([1, 2, 5, 10, 20, EPOCHS]))
epochs_to_show = [e for e in epochs_to_show if e <= EPOCHS]
fig, axes = plt.subplots(1, len(epochs_to_show), figsize=(4 * len(epochs_to_show), 4.5))
for ax, e in zip(np.atleast_1d(axes), epochs_to_show):
    ax.imshow(Image.open(f"{OUT_DIR}/samples/epoch_{e:03d}.png")); ax.set_title(f"Epoch {e}"); ax.axis("off")
plt.tight_layout(); plt.savefig("results/progress_comparison.png", dpi=150); plt.show()

## 7. Final generated faces

In [ ]:
G.eval()
with torch.no_grad():
    faces = G(torch.randn(64, LATENT_DIM, device=device)).cpu()
plt.figure(figsize=(9, 9)); plt.axis("off"); plt.title("Final generated faces (new random noise)")
plt.imshow(vutils.make_grid(faces, nrow=8, padding=2, normalize=True).permute(1, 2, 0))
plt.savefig("results/final_faces.png", dpi=150, bbox_inches="tight"); plt.show()

# Real vs. generated side by side
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(vutils.make_grid(real_batch[:32], nrow=8, normalize=True).permute(1, 2, 0)); ax[0].set_title("Real"); ax[0].axis("off")
ax[1].imshow(vutils.make_grid(faces[:32], nrow=8, normalize=True).permute(1, 2, 0)); ax[1].set_title("Generated"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 8. Latent space interpolation (bonus)
Smooth transitions between faces show the Generator learned a meaningful latent space instead of memorizing images.

In [ ]:
def interpolate(steps=10):
    z1, z2 = torch.randn(1, LATENT_DIM, device=device), torch.randn(1, LATENT_DIM, device=device)
    zs = torch.cat([z1 * (1 - a) + z2 * a for a in torch.linspace(0, 1, steps)])
    with torch.no_grad(): return G(zs).cpu()

rows = torch.cat([interpolate() for _ in range(4)])
plt.figure(figsize=(14, 6)); plt.axis("off"); plt.title("Latent space interpolation")
plt.imshow(vutils.make_grid(rows, nrow=10, normalize=True).permute(1, 2, 0)); plt.show()

## Conclusion
- A DCGAN trained on CelebA generates recognizable 64x64 faces from random noise.
- Sample grids across epochs show the progression from noise blobs, to face-like shapes, to detailed faces.
- Loss curves and D(x) / D(G(z)) scores show the adversarial balance between the two networks.

**Possible improvements:** train longer, use the full dataset, add spectral normalization or WGAN-GP, increase resolution to 128x128, evaluate with FID.

## 9. Webcam demo
Take a photo with your webcam. The app detects and crops your face, then **searches the Generator's latent space for the noise vector that best recreates it** (GAN inversion). You see your face next to the GAN's closest version of you, plus a grid of brand-new random faces.

The camera needs HTTPS, so run the cell below and open the **public Gradio link** it prints (not the inline preview).

In [ ]:
!pip install -q gradio
import cv2, gradio as gr
import torch.nn.functional as F

# Run this section after training (or after loading the checkpoint below)
# ckpt = torch.load(f"{OUT_DIR}/checkpoints/gan_last.pt", map_location=device)
# G.load_state_dict(ckpt["G"])
G.eval()
for p in G.parameters(): p.requires_grad_(False)

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def crop_face(img):
    """Detect the largest face and crop it with CelebA-like framing -> 64x64 RGB."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(80, 80))
    if len(faces) == 0: return None
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    s = int(w * 1.5)
    cx, cy = int(x + w / 2), int(y + h / 2 + 0.1 * h)
    padded = cv2.copyMakeBorder(img, s, s, s, s, cv2.BORDER_REPLICATE)
    x0, y0 = cx - s // 2 + s, cy - s // 2 + s
    crop = padded[y0:y0 + s, x0:x0 + s]
    return cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

def to_display(t, size=256):
    arr = ((t.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()
    arr = cv2.resize(arr, (size, size), interpolation=cv2.INTER_CUBIC)
    return (arr.clip(0, 1) * 255).astype(np.uint8)

def invert(target, steps=300, lr=0.05):
    """Find the latent vector z whose generated image best matches the target face."""
    z = torch.randn(1, LATENT_DIM, device=device, requires_grad=True)
    opt = torch.optim.Adam([z], lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss = F.mse_loss(G(z), target) + 1e-3 * z.pow(2).mean()
        loss.backward(); opt.step()
    return z.detach()

def run_demo(img):
    if img is None: return None, None, None, "Take a photo first."
    crop = crop_face(img)
    if crop is None: return None, None, None, "No face detected. Face the camera in good light and try again."
    target = (torch.from_numpy(crop).permute(2, 0, 1).float() / 127.5 - 1).unsqueeze(0).to(device)
    z = invert(target)
    with torch.no_grad():
        mine = G(z)[0]
        rand = G(torch.randn(16, LATENT_DIM, device=device)).cpu()
    grid = vutils.make_grid(rand, nrow=4, padding=2, normalize=True).permute(1, 2, 0).numpy()
    grid = (cv2.resize(grid, (512, 512), interpolation=cv2.INTER_CUBIC).clip(0, 1) * 255).astype(np.uint8)
    return to_display(target[0]), to_display(mine), grid, "Done."

with gr.Blocks(title="GAN Face Generator") as app:
    gr.Markdown("# GAN Face Generator\nTake a photo, then press **Generate**.")
    cam = gr.Image(sources=["webcam"], type="numpy", label="Webcam")
    btn = gr.Button("Generate", variant="primary")
    with gr.Row():
        o1 = gr.Image(label="Your face (cropped, 64x64)")
        o2 = gr.Image(label="GAN's closest version of you")
        o3 = gr.Image(label="New random GAN faces")
    status = gr.Textbox(label="Status", interactive=False)
    btn.click(run_demo, cam, [o1, o2, o3, status])

app.launch(share=True)